# RLS: роли эквайринга (`rls_acq_user` / `user_filial` / `role_sheet`)

Справочники прав для **одного** дашборда: лист режется флагом роли, РФ — списком филиалов.

Инструкция: `HOW_TO_rls_acq_roles.md`. SQL: `sources/sql/rls_acq_acl.sql`, `sources/sql/acq_rls_virtual_all.sql`.

| Таблица | Смысл |
|---|---|
| `sbx_da.rls_acq_user` | логин → роль, `is_all_filials` |
| `sbx_da.rls_acq_user_filial` | РФ: логин → `filial_filter` (как на дашборде) |
| `sbx_da.rls_acq_role_sheet` | матрица листов из фото |

Старый `rls_acq_dashboard` (только эффективность) **не** удаляем.

Логины = `{{ current_username() }}` в Superset. Impala не нужен.


In [ ]:
import getpass

import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect

drp_schema = 'sbx_da'
drp_superset_grant_role = 'raisa_superset'
NOTEBOOK_REV = '2026-09-24-rls-roles-v2'

# username как в Superset (SQL Lab: SELECT '{{ current_username() }}')
USERS = [
    {
        'username': 'Shestopalov-VYur',
        'role': 'GO_DTPP',
        'is_all_filials': '1',
        'comment': 'ГО ДТПП — полный доступ',
    },
    {
        'username': 'baxhaev-as',
        'role': 'GO_BIZ',
        'is_all_filials': '1',
        'comment': 'ГО Бизнес',
    },
    {
        'username': 'elina-ad',
        'role': 'RF_KM',
        'is_all_filials': '0',
        'comment': 'РФ КМ, Санкт-Петербургский РФ',
    },
]

# Только РФ (is_all_filials = 0). Значение = filial_filter дашборда.
USER_FILIALS = [
    {'username': 'elina-ad', 'filial_filter': 'Санкт-Петербургский РФ'},
]

ROLE_SHEET = [
    {'role': 'GO_DTPP', 'overview': '1', 'tsp_eff': '1', 'pnl': '1', 'clients': '1', 'terminals': '1', 'mcc': '1'},
    {'role': 'GO_BIZ', 'overview': '0', 'tsp_eff': '1', 'pnl': '0', 'clients': '1', 'terminals': '0', 'mcc': '0'},
    {'role': 'RF_ROE', 'overview': '0', 'tsp_eff': '1', 'pnl': '0', 'clients': '1', 'terminals': '1', 'mcc': '0'},
    {'role': 'RF_KM', 'overview': '0', 'tsp_eff': '1', 'pnl': '0', 'clients': '1', 'terminals': '0', 'mcc': '0'},
]

users_df = pd.DataFrame(USERS)
filial_df = pd.DataFrame(USER_FILIALS)
sheet_df = pd.DataFrame(ROLE_SHEET)

for df, key in ((users_df, 'username'), (filial_df, 'username')):
    if len(df):
        df[key] = df[key].astype(str).str.strip()
users_df['role'] = users_df['role'].astype(str).str.strip()
users_df['is_all_filials'] = users_df['is_all_filials'].astype(str).str.strip()
if len(filial_df):
    filial_df['filial_filter'] = filial_df['filial_filter'].astype(str).str.strip()

dup = users_df['username'].str.lower().duplicated(keep=False)
if dup.any():
    raise RuntimeError('Дубли username:\n' + users_df.loc[dup].to_string())

rf_users = set(users_df.loc[users_df['is_all_filials'] != '1', 'username'].str.lower())
fil_users = set(filial_df['username'].str.lower()) if len(filial_df) else set()
missing_fil = rf_users - fil_users
if missing_fil:
    raise RuntimeError('РФ без filial_filter: ' + ', '.join(sorted(missing_fil)))

print('rev', NOTEBOOK_REV)
print('users', len(users_df), 'filial rows', len(filial_df), 'roles', len(sheet_df))
display(users_df)
display(filial_df)
display(sheet_df)


## 1) DRP login


In [ ]:
drp_user = input('DRP user: ').strip()
drp_password = getpass.getpass('DRP password: ')
drp = connect(
    to='DRP',
    user_params={'user_name': drp_user, 'password': drp_password},
)
print('DRP connected as', drp_user)


## 2) DROP / CREATE / GRANT

CREATE без кавычек колонок — иначе GP через jaydebeapi падает на `"`.


In [ ]:
def pg_ident(name):
    return '"' + str(name).replace('"', '""') + '"'


def upload_text_table(fq, df, col_sql):
    upload_df = df.copy()
    upload_df.columns = [str(c).strip().lower() for c in upload_df.columns]
    for c in upload_df.columns:
        upload_df[c] = upload_df[c].map(lambda x: None if pd.isna(x) else str(x)).astype(object)
    create_sql = 'CREATE TABLE ' + fq + ' (' + col_sql + ')'
    print('CREATE SQL:', create_sql)
    drp.execute('DROP TABLE IF EXISTS ' + fq)
    drp.execute(create_sql)
    if len(upload_df):
        drp.write(table=fq, df=upload_df, mode='append')
    cnt = drp.fetch('SELECT COUNT(*) AS n FROM ' + fq)
    n = int(pd.to_numeric(cnt.iloc[0, 0], errors='coerce'))
    print('OK', fq, 'rows=', n)
    return n


tables = {
    drp_schema + '.rls_acq_user': (
        users_df,
        'username TEXT, role TEXT, is_all_filials TEXT, comment TEXT',
    ),
    drp_schema + '.rls_acq_user_filial': (
        filial_df if len(filial_df) else pd.DataFrame(columns=['username', 'filial_filter']),
        'username TEXT, filial_filter TEXT',
    ),
    drp_schema + '.rls_acq_role_sheet': (
        sheet_df,
        'role TEXT, overview TEXT, tsp_eff TEXT, pnl TEXT, clients TEXT, terminals TEXT, mcc TEXT',
    ),
}

grant_roles = [drp_superset_grant_role, 'Shestopalov-VYur']
# Если SQL Lab: permission denied for relation rls_acq_user —
# SELECT current_user в Lab и добавь этот логин сюда, перегони ячейку.

with drp:
    for fq, (df, cols) in tables.items():
        upload_text_table(fq, df, cols)
    drp.execute('GRANT USAGE ON SCHEMA ' + drp_schema + ' TO raisa_superset')
    for fq in tables:
        for role in grant_roles:
            sql = 'GRANT SELECT ON TABLE ' + fq + ' TO ' + pg_ident(role)
            try:
                drp.execute(sql)
                print('OK', sql)
            except Exception as exc:
                print('FAIL GRANT', role, fq, type(exc).__name__, str(exc)[:240])

    chk_u = drp.fetch('SELECT username, role, is_all_filials FROM ' + drp_schema + '.rls_acq_user ORDER BY 1')
    chk_f = drp.fetch('SELECT username, filial_filter FROM ' + drp_schema + '.rls_acq_user_filial ORDER BY 1, 2')
    chk_s = drp.fetch('SELECT * FROM ' + drp_schema + '.rls_acq_role_sheet ORDER BY 1')

display(chk_u)
display(chk_f)
display(chk_s)


## 3) Smoke: какие листы откроются

Без Jinja: подставляем логин. `sheet_ok` = флаг роли; для РФ смотри `filial_n`.


In [ ]:
smoke_sql = '''
SELECT
  u.username,
  u.role,
  u.is_all_filials,
  s.overview,
  s.tsp_eff,
  s.pnl,
  s.clients,
  s.terminals,
  s.mcc,
  (SELECT COUNT(*) FROM sbx_da.rls_acq_user_filial f
   WHERE lower(BTRIM(f.username)) = lower(BTRIM(u.username))) AS filial_n
FROM sbx_da.rls_acq_user u
JOIN sbx_da.rls_acq_role_sheet s
  ON BTRIM(CAST(s.role AS TEXT)) = BTRIM(CAST(u.role AS TEXT))
ORDER BY u.username
'''

expect = {
    'GO_DTPP': {'overview': '1', 'tsp_eff': '1', 'pnl': '1', 'clients': '1', 'terminals': '1', 'mcc': '1', 'all_fil': True},
    'GO_BIZ': {'overview': '0', 'tsp_eff': '1', 'pnl': '0', 'clients': '1', 'terminals': '0', 'mcc': '0', 'all_fil': True},
    'RF_ROE': {'overview': '0', 'tsp_eff': '1', 'pnl': '0', 'clients': '1', 'terminals': '1', 'mcc': '0', 'all_fil': False},
    'RF_KM': {'overview': '0', 'tsp_eff': '1', 'pnl': '0', 'clients': '1', 'terminals': '0', 'mcc': '0', 'all_fil': False},
}

with drp:
    smoke = drp.fetch(smoke_sql)
display(smoke)

ok = True
for _, row in smoke.iterrows():
    role = str(row['role']).strip()
    exp = expect.get(role)
    if not exp:
        print('UNKNOWN ROLE', role)
        ok = False
        continue
    for col in ('overview', 'tsp_eff', 'pnl', 'clients', 'terminals', 'mcc'):
        got = str(row[col]).strip()
        if got != exp[col]:
            print('MISMATCH', row['username'], col, 'got', got, 'want', exp[col])
            ok = False
    all_fil = str(row['is_all_filials']).strip() in ('1', 'true', 'True', 'Y', 'y')
    if all_fil != exp['all_fil']:
        print('MISMATCH is_all_filials', row['username'], all_fil, 'want', exp['all_fil'])
        ok = False
    if not all_fil and int(row['filial_n']) < 1:
        print('RF without filial', row['username'])
        ok = False

print('SMOKE', 'OK' if ok else 'FAIL')
print('Дальше: HOW_TO_rls_acq_roles.md — virtual dataset на копии дашборда.')
print('SQL Lab: SELECT current_username через Jinja, сверить с username в ACL.')
